In [1]:
# Setup
# %pip install -q openai python-dotenv


## 1) Import


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json


## 2) Load environment variables - please read instructions carefully


In [3]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
# load_dotenv()

In [4]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

In [5]:
# llm_name = "gpt-4"
# openai_key = os.getenv("OPENAI_API_KEY")
# client = OpenAI(api_key=openai_key)

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # placeholder, Ollama ignores this
)

llm_name = "qwen3"


## 3) Tool


In [6]:
# Implement the functions actions
import math
def calculate(expression: str) -> str:
    try: 
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result) 
    except Exception as e: 
        return f"Error: {e}"

known_actions = {"calculate": calculate}


## 4) Tool Schema


In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calculate the result of a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression with basic mathematical operators or python's math library to evaluate, e.g., 'math.sqrt(16)'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

In [8]:
system_prompt = """
You run in a loop of Thought, Action, Observation.
At the end of the loop you output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you.
Observation will be the result of running those actions.

Important notes:
Do not invent any facts, numbers. Use available actions instead.

""".strip()


## 5) Custom Agent


In [9]:
# Create an agent using OpenAI native tool calling
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({
            "role": "user",
            "content": message
        })
        count =0;
        while True:
            response = client.chat.completions.create(
                model=llm_name,
                temperature=0.4,
                messages=self.messages,
                tools=tools
            )
            print("\n turn ", count)
            print("\n messages sent to LLM ")
            print("\n", self.messages)
            print("\n response from LLM ")
            print("\n", response)

            assistant_message = response.choices[0].message
            self.messages.append(assistant_message)

            finish_reason = response.choices[0].finish_reason
            print("\n stop_reason", finish_reason)

            if finish_reason=="stop":
                return assistant_message.content

            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                tool_result = known_actions[tool_name](**tool_args)

                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })
            count=count+1


In [10]:
agent = Agent(system_prompt)
question = "What is the square root of 144?"
response = agent(question)
print(response)


 turn  0

 messages sent to LLM 

 [{'role': 'system', 'content': 'You run in a loop of Thought, Action, Observation.\nAt the end of the loop you output an Answer.\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you.\nObservation will be the result of running those actions.\n\nImportant notes:\nDo not invent any facts, numbers. Use available actions instead.'}, {'role': 'user', 'content': 'What is the square root of 144?'}]

 response from LLM 

 ChatCompletion(id='chatcmpl-148', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_gdmrseqi', function=Function(arguments='{"expression":"math.sqrt(144)"}', name='calculate'), type='function', index=0)], reasoning="Okay, the user is asking for the square root of 1

In [11]:
agent = Agent(system_prompt)
question = "What is the square of 13?"
response = agent(question)
print(response)


 turn  0

 messages sent to LLM 

 [{'role': 'system', 'content': 'You run in a loop of Thought, Action, Observation.\nAt the end of the loop you output an Answer.\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you.\nObservation will be the result of running those actions.\n\nImportant notes:\nDo not invent any facts, numbers. Use available actions instead.'}, {'role': 'user', 'content': 'What is the square of 13?'}]

 response from LLM 

 ChatCompletion(id='chatcmpl-381', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ct7k812f', function=Function(arguments='{"expression":"math.pow(13, 2)"}', name='calculate'), type='function', index=0)], reasoning="Okay, the user is asking for the square of 13. Let me 

In [ ]:
agent = Agent(system_prompt)
question = "create a table: first column is 1 to 10, second column is 2 to the power of the first column"
response = agent(question)
print(response)


 turn  0

 messages sent to LLM 

 [{'role': 'system', 'content': 'You run in a loop of Thought, Action, Observation.\nAt the end of the loop you output an Answer.\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you.\nObservation will be the result of running those actions.\n\nImportant notes:\nDo not invent any facts, numbers. Use available actions instead.'}, {'role': 'user', 'content': 'create a table: first column is 1 to 10, second column is 2 to the power of the first column'}]

 response from LLM 

 ChatCompletion(id='chatcmpl-279', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_akr2340r', function=Function(arguments='{"expression":"2 ** 1"}', name='calculate'), type='function', index=0), ChatComp